# Main — Barlow Twins → S5 → classification

이 노트북은 논문 파이프라인 전체를 한곳에 둔다.

1. Barlow Twins로 인코더를 사전학습한다.
2. 그 인코더로 S5(VAE, B0–B3 skip)를 초기화하고 분할을 학습한다.
3. 분할기를 고정하고 `LesionPresenceHead`만 학습한다. 확률이 τ보다 낮으면 마스크를 비운다.

`RUN_BT`, `RUN_SEG`, `RUN_CLS`를 True로 바꾸면 각 단계가 실행된다. 이미 체크포인트가 있으면 건너뛴다.


# Main — Barlow Twins → S5 → classification

This notebook holds the full paper pipeline in one place.

1. Pre-train the encoder with Barlow Twins.
2. Initialize S5 (VAE, B0–B3 skips) from that encoder and train segmentation.
3. Freeze the segmenter and train only `LesionPresenceHead`. Clear the mask when the probability is below τ.

Set `RUN_BT`, `RUN_SEG`, or `RUN_CLS` to True to run a stage. An existing checkpoint is skipped.


In [ ]:
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

from pipeline.config import RESULT_DIR, CKPT_DIR, BT_BEST, BT_LAST
from pipeline.data import datasets
from pipeline.engine import (
    DEVICE,
    atomic_torch_save,
    batch_dice,
    batch_pred_gt_sums,
    safe_load,
    seg_step_loss,
)
from pipeline.metrics import binary_auroc, metrics_bundle
from pipeline.stage_segment import make_bundle

RATIO = 0.3
BASE = 32

BT_EPOCHS = 200
BT_LR = 3e-4
BT_BATCH = 16

SEG_EPOCHS = 400
SEG_LR = 1e-4
SEG_BATCH = 4
SEG_ES = 40

CLS_EPOCHS = 40
CLS_ES = 10
CLS_POS_WEIGHT = 2.0

RUN_BT = False   # True: Barlow Twins pre-training
RUN_SEG = False  # True: S5 segmentation
RUN_CLS = False  # True: presence head + gate

print("device", DEVICE)
print("BT", BT_BEST)
print("results", RESULT_DIR)


## 모델

인코더는 Barlow Twins와 분할이 공유한다. S5 디코더는 μ와 B0–B3 skip으로 마스크를 그린다. B4는 쓰지 않는다. 분류 헤드는 skip을 보지 않고 μ만 받는다.


## Models

The encoder is shared by Barlow Twins and segmentation. The S5 decoder draws the mask from μ and skips B0–B3. B4 is unused. The classification head does not see skips. It reads only μ.


In [ ]:
class ResBlockUp(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super().__init__()
        self.act = act
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.BatchNorm2d(filters_out),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)
        if self.act:
            conv2 = self.lrelu(conv2)
        conv3 = self.conv3_block(x)
        if self.act:
            conv3 = self.lrelu(conv3)
        return conv2 + conv3


class ResBlockDown(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super().__init__()
        self.act = act
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)
        if self.act:
            conv2 = self.lrelu(conv2)
        conv3 = self.conv3_block(x)
        if self.act:
            conv3 = self.lrelu(conv3)
        return conv2 + conv3


class ConcatConvFuse(nn.Module):
    def __init__(self, dec_ch: int, skip_ch: int):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv2d(dec_ch + skip_ch, dec_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        if skip.shape[-2:] != x.shape[-2:]:
            skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.fuse(torch.cat([x, skip], dim=1))


class Encoder(nn.Module):
    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        self.encoder_initial_flair = self._create_initial_encoder(1, base, dropout_rate)
        self.encoder_features_flair = self._create_feature_encoder(base, dropout_rate)
        self.encoder_mu_flair = self._create_mu_encoder(32 * base)
        self.encoder_logvar_flair = self._create_logvar_encoder(32 * base)

    def _create_initial_encoder(self, in_channels, base, dropout_rate):
        return nn.Sequential(
            nn.Conv2d(in_channels, base, 3, stride=1, padding=1),
            nn.BatchNorm2d(base),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(p=dropout_rate),
        )

    def _create_feature_encoder(self, base, dropout_rate):
        return nn.Sequential(
            ResBlockDown(base, 2 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(2 * base, 4 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(4 * base, 8 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(8 * base, 16 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(16 * base, 32 * base),
            nn.Dropout2d(p=dropout_rate),
        )

    def _create_mu_encoder(self, C):
        return nn.Sequential(nn.Conv2d(C, C, 1), nn.BatchNorm2d(C))

    def _create_logvar_encoder(self, C):
        return nn.Sequential(nn.Conv2d(C, C, 1), nn.BatchNorm2d(C))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward_features(self, flair):
        e0 = self.encoder_initial_flair(flair)
        e1 = self.encoder_features_flair[0](e0)
        e1d = self.encoder_features_flair[1](e1)
        e2 = self.encoder_features_flair[2](e1d)
        e2d = self.encoder_features_flair[3](e2)
        e3 = self.encoder_features_flair[4](e2d)
        e3d = self.encoder_features_flair[5](e3)
        e4 = self.encoder_features_flair[6](e3d)
        e4d = self.encoder_features_flair[7](e4)
        h = self.encoder_features_flair[8](e4d)
        h = self.encoder_features_flair[9](h)
        mu = self.encoder_mu_flair(h)
        logvar = self.encoder_logvar_flair(h)
        z = self.reparameterize(mu, logvar)
        return mu, logvar, z, (e0, e1, e2, e3, e4)

    def forward(self, flair):
        mu, logvar, z, _ = self.forward_features(flair)
        return mu, logvar, z


class Decoder(nn.Module):
    """S5: B0..B3 concat+3x3 fuse, B4 off."""

    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        feat = 32 * base
        self.up1 = ResBlockUp(feat, feat)
        self.drop1 = nn.Dropout2d(p=dropout_rate)
        self.up2 = ResBlockUp(feat, feat)
        self.drop2 = nn.Dropout2d(p=dropout_rate)
        self.up3 = ResBlockUp(feat, 16 * base)
        self.drop3 = nn.Dropout2d(p=dropout_rate)
        self.up4 = ResBlockUp(16 * base, 8 * base)
        self.drop4 = nn.Dropout2d(p=dropout_rate)
        self.up5 = ResBlockUp(8 * base, base)
        self.drop5 = nn.Dropout2d(p=dropout_rate)
        self.out_conv = nn.Conv2d(base, 1, 3, stride=1, padding=1)
        self.fuse3 = ConcatConvFuse(feat, 8 * base)
        self.fuse2 = ConcatConvFuse(16 * base, 4 * base)
        self.fuse1 = ConcatConvFuse(8 * base, 2 * base)
        self.fuse0 = ConcatConvFuse(base, base)

    def forward(self, z, skips):
        e0, e1, e2, e3, e4 = skips
        del e4
        x = self.drop1(self.up1(z))
        x = self.drop2(self.up2(x))
        x = self.fuse3(x, e3)
        x = self.drop3(self.up3(x))
        x = self.fuse2(x, e2)
        x = self.drop4(self.up4(x))
        x = self.fuse1(x, e1)
        x = self.drop5(self.up5(x))
        x = self.fuse0(x, e0)
        return self.out_conv(x)


class VAE(nn.Module):
    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(base, dropout_rate)
        self.decoder = Decoder(base, dropout_rate)

    def forward(self, x):
        mu, logvar, z, skips = self.encoder.forward_features(x)
        out = self.decoder(z, skips)
        return out, mu, logvar


class LesionPresenceHead(nn.Module):
    def __init__(self, in_channels: int, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(in_channels * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, feature: torch.Tensor) -> torch.Tensor:
        pooled = torch.cat([
            self.avg_pool(feature).flatten(1),
            self.max_pool(feature).flatten(1),
        ], dim=1)
        return self.classifier(pooled).squeeze(1)


## 1. Barlow Twins 사전학습

같은 슬라이스의 90° / 270° 두 뷰를 공유 인코더에 넣는다. μ와 logvar로 교차상관과 KL 손실을 만든다. BN은 고정한다.


## 1. Barlow Twins pre-training

Two views of the same slice (90° / 270°) enter a shared encoder. The loss uses cross-correlation on μ and a KL term on μ, logvar. BatchNorm stays frozen.


In [ ]:
def freeze_bn(encoder: nn.Module):
    for m in encoder.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False


def kl_gaussian(mu1, sigma1, mu2, sigma2, eps: float = 1e-12):
    var1 = (sigma1 ** 2).clamp_min(eps)
    var2 = (sigma2 ** 2).clamp_min(eps)
    diff2 = (mu1 - mu2) ** 2
    return 0.5 * (torch.log(var2) - torch.log(var1) + (var1 + diff2) / var2 - 1.0)


def kl_matrix_loss_weighted(K, w_diag=1.0, w_off=1.0, target_off=3.0):
    eye = torch.eye(K.size(0), device=K.device, dtype=K.dtype).bool()
    diag_loss = (K.diagonal() ** 2).mean()
    off_loss = ((K[~eye] - target_off) ** 2).mean()
    return w_diag * diag_loss + w_off * off_loss


def cross_corr_mu_loss(m1v, m2v, eps=1e-12, lambda_off=1.0):
    B = m1v.size(0)
    z1n = (m1v - m1v.mean(0)) / (m1v.std(0) + eps)
    z2n = (m2v - m2v.mean(0)) / (m2v.std(0) + eps)
    C = (z1n.T @ z2n) / float(B)
    D = C.size(0)
    diag = torch.diag(C)
    off = C - torch.diag(diag)
    diag_loss = (1.0 - diag).pow(2).mean()
    off_loss = off.pow(2).sum() / (D * (D - 1) + eps)
    return diag_loss + lambda_off * off_loss, C


def bt_batch_loss(encoder, view1, view2, ep: int):
    if view1.ndim == 3:
        view1 = view1.unsqueeze(1)
    if view2.ndim == 3:
        view2 = view2.unsqueeze(1)
    m1, v1_, z1 = encoder(view1)
    m2, v2_, z2 = encoder(view2)
    v1_ = v1_.clamp(-5, 2)
    v2_ = v2_.clamp(-5, 2)
    gap = nn.AdaptiveAvgPool2d(1)
    s = lambda x: gap(x).flatten(1) if x.dim() == 4 else x
    m1v, v1v = s(m1), s(v1_)
    m2v, v2v = s(m2), s(v2_)
    sig1 = (0.5 * v1v).exp()
    sig2 = (0.5 * v2v).exp()
    K12 = kl_gaussian(m1v.unsqueeze(2), sig1.unsqueeze(2), m2v.unsqueeze(1), sig2.unsqueeze(1))
    K21 = kl_gaussian(m2v.unsqueeze(2), sig2.unsqueeze(2), m1v.unsqueeze(1), sig1.unsqueeze(1))
    K = 0.5 * (K12 + K21).mean(dim=0)
    tgt = 1.0 + min(1.0, ep / 100.0)
    loss_kl = kl_matrix_loss_weighted(K, w_diag=1.0, w_off=1.0, target_off=tgt)
    loss_cmu, _ = cross_corr_mu_loss(m1v, m2v, lambda_off=1.0)
    return loss_kl + 0.2 * loss_cmu


def load_bt_into_encoder(encoder: nn.Module, bt_ckpt: Path):
    st = safe_load(bt_ckpt, map_location="cpu")
    enc_state = st["encoder"] if isinstance(st, dict) and "encoder" in st else st
    enc_state = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in enc_state.items()}
    missing, unexpected = encoder.load_state_dict(enc_state, strict=False)
    print(f"encoder load missing={len(missing)} unexpected={len(unexpected)}")


def train_bt():
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    if BT_BEST.exists():
        print("skip BT", BT_BEST)
        return BT_BEST
    data = datasets.covid(batch_size=BT_BATCH, num_workers=0, pin_memory=False)
    encoder = Encoder(base=BASE).to(DEVICE)
    freeze_bn(encoder)
    opt = torch.optim.AdamW(encoder.parameters(), lr=BT_LR, weight_decay=1e-4)
    best_val = float("inf")
    for ep in range(1, BT_EPOCHS + 1):
        encoder.train()
        freeze_bn(encoder)
        running = 0.0
        for batch in data.train_loader:
            view1 = batch[2].to(DEVICE)
            view2 = batch[3].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = bt_batch_loss(encoder, view1, view2, ep)
            loss.backward()
            nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
            opt.step()
            running += float(loss.item())
        encoder.eval()
        val_running = 0.0
        with torch.no_grad():
            for batch in data.val_loader:
                view1 = batch[2].to(DEVICE)
                view2 = batch[3].to(DEVICE)
                val_running += float(bt_batch_loss(encoder, view1, view2, ep).item())
        train_avg = running / max(1, len(data.train_loader))
        val_avg = val_running / max(1, len(data.val_loader))
        print(f"[BT] ep {ep}/{BT_EPOCHS} train={train_avg:.6f} val={val_avg:.6f}")
        payload = {"ep": ep, "encoder": encoder.state_dict(), "best_val": best_val}
        atomic_torch_save(payload, BT_LAST)
        if val_avg < best_val - 1e-6:
            best_val = val_avg
            payload["best_val"] = best_val
            atomic_torch_save(payload, BT_BEST)
    print("BT done", BT_BEST if BT_BEST.exists() else BT_LAST)
    return BT_BEST if BT_BEST.exists() else BT_LAST


if RUN_BT:
    train_bt()


## 2. S5 분할 학습

Barlow Twins 인코더로 초기화한 뒤, 인코더와 디코더를 함께 학습한다. 손실은 Dice+BCE에 KL을 더한 것이다. 디코드는 μ로 고정한다. 검증 손실이 40 에폭 동안 줄지 않으면 멈춘다.


## 2. S5 segmentation

Initialize from the Barlow Twins encoder, then train the encoder and decoder together. The loss mixes Dice and BCE, then adds KL. Decoding uses μ. Training stops when the validation loss does not fall for 40 epochs.


In [ ]:
def train_s5(ratio=RATIO):
    pct = int(round(ratio * 100))
    out_dir = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5"
    out_dir.mkdir(parents=True, exist_ok=True)
    best_path = out_dir / "seg_best.pt"
    last_path = out_dir / "seg_last.pt"
    report_path = out_dir / "report_seg.json"
    if best_path.exists() and report_path.exists():
        print("skip SEG", best_path)
        return best_path

    _full, train_loader, val_loader, test_loader, n_train = make_bundle(ratio, SEG_BATCH)
    model = VAE(base=BASE).to(DEVICE)
    bt = BT_BEST if BT_BEST.exists() else BT_LAST
    if not last_path.exists():
        if not bt.exists():
            raise FileNotFoundError(f"BT checkpoint missing: {bt}. Set RUN_BT=True first.")
        load_bt_into_encoder(model.encoder, bt)

    opt = torch.optim.AdamW(model.parameters(), lr=SEG_LR, weight_decay=1e-4)
    sched = ReduceLROnPlateau(opt, mode="min", factor=0.95, patience=3, min_lr=1e-8)
    best_loss, best_ep, bad = float("inf"), 0, 0
    start_ep = 1
    if last_path.exists():
        st = safe_load(last_path, map_location="cpu")
        model.load_state_dict(st["model"])
        start_ep = int(st.get("ep", 0)) + 1
        best_loss = float(st.get("best_loss", best_loss))
        best_ep = int(st.get("best_ep", 0))
        bad = max(0, start_ep - 1 - best_ep)
        print(f"resume S5 from ep {start_ep}")

    for ep in range(start_ep, SEG_EPOCHS + 1):
        model.train()
        tr_loss, ntr = 0.0, 0
        for batch in train_loader:
            x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss, _dice, _kl = seg_step_loss(model, x, y)
            loss.backward()
            opt.step()
            tr_loss += float(loss.item())
            ntr += 1
        model.eval()
        va_loss, nva = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
                loss, _, _ = seg_step_loss(model, x, y)
                va_loss += float(loss.item())
                nva += 1
        va_loss /= max(1, nva)
        sched.step(va_loss)
        print(f"[S5 r{pct} {ep}/{SEG_EPOCHS}] train={tr_loss / max(1, ntr):.4f} val={va_loss:.4f}")
        payload = {
            "model": model.state_dict(),
            "ep": ep,
            "best_loss": best_loss,
            "best_ep": best_ep,
            "ratio": ratio,
            "mode": "s5",
        }
        atomic_torch_save(payload, last_path)
        if va_loss < best_loss - 1e-5:
            best_loss, best_ep, bad = va_loss, ep, 0
            payload["best_loss"] = best_loss
            payload["best_ep"] = best_ep
            atomic_torch_save(payload, best_path)
        else:
            bad += 1
            if bad >= SEG_ES:
                print(f"early stop S5 @ {ep}")
                break

    model.load_state_dict(safe_load(best_path, map_location="cpu")["model"])
    model.eval()
    dices, gts, preds = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
            if x.ndim == 3:
                x = x.unsqueeze(1)
            if y.ndim == 3:
                y = y.unsqueeze(1)
            mu, _lv, _z, skips = model.encoder.forward_features(x)
            logits = model.decoder(mu, skips)
            dices.append(batch_dice(logits, y))
            ps, gs = batch_pred_gt_sums(logits, y)
            preds.append(ps)
            gts.append(gs)
    dices = np.concatenate(dices)
    gts = np.concatenate(gts)
    preds = np.concatenate(preds)
    nonempty = gts > 0
    empty = ~nonempty
    report = {
        "ratio": ratio,
        "mode": "s5",
        "n_train": n_train,
        "best_ep": best_ep,
        "best_val_loss": best_loss,
        "test": {
            "dice": float(dices.mean()),
            "nonempty_dice": float(dices[nonempty].mean()) if nonempty.any() else None,
            "empty_fp_rate": float((preds[empty] > 0).mean()) if empty.any() else None,
            "n": int(len(dices)),
        },
    }
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(report)
    return best_path


if RUN_SEG:
    train_s5(RATIO)


## 3. 병변 존재 분류

학습이 끝난 S5는 고정한다. μ에 헤드를 달아 존재 확률을 낸다. 검증 Dice가 최대인 τ로 테스트 마스크를 게이트한다.


## 3. Lesion-presence classification

The trained S5 is frozen. A head on μ estimates the presence probability. Test masks are gated by the τ that maximizes validation Dice.


In [ ]:
def prep_xy(batch):
    x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
    if x.ndim == 3:
        x = x.unsqueeze(1)
    if y.ndim == 3:
        y = y.unsqueeze(1)
    return x, y


@torch.inference_mode()
def collect(model, head, loader):
    model.eval()
    head.eval()
    bags = {"gt": [], "mass": [], "dice": [], "score": []}
    for batch in loader:
        x, y = prep_xy(batch)
        mu, _lv, _z, skips = model.encoder.forward_features(x)
        seg_logits = model.decoder(mu, skips)
        mass, gt = batch_pred_gt_sums(seg_logits, y)
        bags["gt"].append(gt)
        bags["mass"].append(mass)
        bags["dice"].append(batch_dice(seg_logits, y))
        bags["score"].append(torch.sigmoid(head(mu)).detach().cpu().numpy())
    return {k: np.concatenate(v) for k, v in bags.items()}


def pick_tau(val):
    best = None
    for i in range(1, 100):
        thr = i / 100.0
        keep = val["score"] >= thr
        m = metrics_bundle(val["gt"], val["mass"], val["dice"], keep)
        key = (m["overall_dice"], -m["empty_fp_rate"])
        if best is None or key > best["key"]:
            best = {"thr": thr, "key": key, "m": m}
    return best["thr"], best["m"]


def train_s5_cls(ratio=RATIO):
    pct = int(round(ratio * 100))
    seg_ckpt = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5" / "seg_best.pt"
    out_dir = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5_cls"
    out_dir.mkdir(parents=True, exist_ok=True)
    report_path = out_dir / "report.json"
    if report_path.exists():
        print("skip CLS", report_path)
        return json.loads(report_path.read_text(encoding="utf-8"))
    if not seg_ckpt.exists():
        raise FileNotFoundError(f"S5 checkpoint missing: {seg_ckpt}. Set RUN_SEG=True first.")

    _full, train_loader, val_loader, test_loader, n_train = make_bundle(ratio, SEG_BATCH)
    model = VAE(base=BASE).to(DEVICE)
    model.load_state_dict(safe_load(seg_ckpt, map_location="cpu")["model"])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    head = LesionPresenceHead(BASE * BASE).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CLS_POS_WEIGHT], device=DEVICE))
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auroc, bad, best_ep = -1.0, 0, 0
    best_path = out_dir / "head_best.pt"
    trend = []

    for ep in range(1, CLS_EPOCHS + 1):
        head.train()
        total, n = 0.0, 0
        for batch in train_loader:
            x, y = prep_xy(batch)
            with torch.no_grad():
                mu, _lv, _z, _sk = model.encoder.forward_features(x)
            y_cls = (y.flatten(1).sum(1) > 0).float()
            opt.zero_grad(set_to_none=True)
            loss = bce(head(mu), y_cls)
            loss.backward()
            opt.step()
            total += float(loss.item()) * y_cls.size(0)
            n += int(y_cls.size(0))
        val = collect(model, head, val_loader)
        auroc = binary_auroc(val["gt"] > 0, val["score"])
        trend.append({"ep": ep, "train_bce": total / max(n, 1), "val_auroc": auroc})
        print(f"[CLS] ep {ep} bce={trend[-1]['train_bce']:.4f} auroc={auroc:.4f}")
        if auroc > best_auroc + 1e-4:
            best_auroc, bad, best_ep = auroc, 0, ep
            atomic_torch_save({"head": head.state_dict(), "ep": ep, "auroc": auroc}, best_path)
        else:
            bad += 1
            if bad >= CLS_ES:
                break

    head.load_state_dict(safe_load(best_path, map_location="cpu")["head"])
    val = collect(model, head, val_loader)
    test = collect(model, head, test_loader)
    tau, _val_m = pick_tau(val)
    mt = metrics_bundle(test["gt"], test["mass"], test["dice"], test["score"] >= tau)
    report = {
        "ratio": ratio,
        "n_train": n_train,
        "loss": "BCEWithLogits",
        "probability": "p = sigmoid(Head(mu)) = P(lesion present | slice)",
        "tau": tau,
        "best_ep": best_ep,
        "val_auroc": best_auroc,
        "test_dice": float(mt["overall_dice"]),
        "test_nonempty_dice": float(mt["nonempty_dice"]),
        "test_empty_fp_rate": float(mt["empty_fp_rate"]),
    }
    (out_dir / "prob_trend.json").write_text(json.dumps(trend, indent=2), encoding="utf-8")
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(report)
    return report


if RUN_CLS:
    train_s5_cls(RATIO)
